# RL Challenge: gemeinsamer Colab-Durchlauf

Dieses Notebook führt die technische Kette einmal vollständig aus:

`Repository → Installation → ROM-Test → Baselines → Validator → Browser-Runner → DQN → ONNX`

Führen Sie die Zellen der Reihe nach aus. Für den gemeinsamen 128-Schritte-Test ist keine GPU erforderlich. Colab-Speicher ist temporär; laden Sie Modelle herunter, die Sie behalten möchten.

## 1. Repository laden

Die Zelle kann erneut ausgeführt werden. Eine vorhandene Kopie wird dann aktualisiert.

In [ ]:
%cd /content
!test -d rl-challenge || git clone --depth 1 https://github.com/georgmerz/rl-challenge.git
%cd /content/rl-challenge
!git pull --ff-only


## 2. Pakete und Atari-ROMs installieren

Colab bringt PyTorch bereits mit. `pip` installiert nur fehlende oder nicht passende Pakete. AutoROM installiert anschließend die Pong-ROM. Dieser Schritt dauert typischerweise einige Minuten.

In [ ]:
!python -m pip install --quiet --prefer-binary -r requirements.txt
!AutoROM --accept-license > /tmp/autorom.log
!tail -n 3 /tmp/autorom.log


## 3. Installation prüfen

Diese Zelle zeigt die wichtigsten Versionen und öffnet Pong einmal mit Seed 42.

In [ ]:
import sys
import numpy as np
import onnxruntime as ort
import torch
from pettingzoo.atari import pong_v3

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('PyTorch:', torch.__version__)
print('ONNX Runtime:', ort.__version__)
print('CUDA verfügbar:', torch.cuda.is_available())

env = pong_v3.parallel_env()
env.reset(seed=42)
env.close()
print('Setup und Pong-ROM: OK')


## 4. Baselines erzeugen und validieren

Beide Modelle erfüllen denselben ONNX-Vertrag. Die statische Policy ist absichtlich schwach; die Ball-Tracking-Policy reagiert auf Bildpixel.

In [ ]:
!python create_baselines.py
!python validate_submission.py baselines/baseline_pong_static.onnx
!python validate_submission.py baselines/baseline_pong_track_ball.onnx


## 5. Baselines herunterladen

Der Browser-Runner verwendet normale Datei-Uploads. Laden Sie deshalb beide Baselines auf Ihren Rechner herunter und wählen Sie sie anschließend im Runner aus.

In [ ]:
from google.colab import files

files.download('baselines/baseline_pong_static.onnx')
files.download('baselines/baseline_pong_track_ball.onnx')


## 6. Browser-Runner in Colab öffnen

Die nächste Zelle startet `app.py` im Hintergrund und zeigt die Oberfläche direkt im Notebook. Laden Sie die beiden soeben heruntergeladenen Modelle hoch und testen Sie Start, Pause, Fortsetzen und Stop.

Falls das iframe nach einem Runtime-Neustart leer bleibt, führen Sie diese Zelle erneut aus.

In [ ]:
import os
import socket
import subprocess
import sys
import time
from google.colab import output

if 'runner_process' in globals() and runner_process.poll() is None:
    runner_process.terminate()
    runner_process.wait(timeout=5)

runner_environment = os.environ.copy()
runner_environment['PORT'] = '5001'
runner_process = subprocess.Popen(
    [sys.executable, 'app.py'],
    env=runner_environment,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

for _ in range(50):
    with socket.socket() as connection:
        if connection.connect_ex(('127.0.0.1', 5001)) == 0:
            break
    time.sleep(0.1)
else:
    raise RuntimeError('Der Browser-Runner konnte nicht gestartet werden.')

output.serve_kernel_port_as_iframe(5001, height=720)


## 7. DQN-, Checkpoint- und ONNX-Weg testen

128 Schritte prüfen die gesamte Pipeline, trainieren aber noch keine sinnvolle Policy. DQN-Updates beginnen im Beispiel erst nach 5.000 Schritten.

In [ ]:
!python train_dqn_vanilla.py --timesteps 128 --export-every 128 --output baselines/lecture_smoke.onnx
!python validate_submission.py baselines/lecture_smoke.onnx


Die erzeugte ONNX-Datei kann nun heruntergeladen und ebenfalls im Browser-Runner getestet werden.

In [ ]:
files.download('baselines/lecture_smoke.onnx')


## 8. Optional: länger trainieren

Setzen Sie `RUN_LONG_TRAINING = True`, wenn genügend Zeit vorhanden ist. 100.000 Schritte führen tatsächlich DQN-Updates aus, garantieren aber noch keine starke Policy. Für GPU-Training wählen Sie in Colab **Runtime → Change runtime type → GPU**.

In [ ]:
RUN_LONG_TRAINING = False

if RUN_LONG_TRAINING:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    !python train_dqn_vanilla.py --device {device} --timesteps 100000 --opponent baselines/baseline_pong_track_ball.onnx --output baselines/baseline_pong_dqn.onnx
    !python validate_submission.py baselines/baseline_pong_dqn.onnx
    files.download('baselines/baseline_pong_dqn.onnx')
else:
    print('Langes Training übersprungen. Setzen Sie RUN_LONG_TRAINING auf True, um es zu starten.')


## Ergebnis

Sie haben Baselines erzeugt, den ONNX-Vertrag validiert, ein Duell im Runner gestartet und den DQN-Exportweg ausgeführt. Für die Projektarbeit ersetzen oder erweitern Sie nun bewusst einzelne Entscheidungen des Beispiels und evaluieren Ihre Varianten unter identischen Bedingungen.